# 18 Gold Procedure CarePlan Summary

## Purpose

This notebook creates a patient-level treatment and care management analytics table.

## What We Are Doing

We will combine:
- Procedure data
- CarePlan data

to measure:
- treatment intensity
- procedure burden
- care management burden
- active care plans
- completed care plans
- care coordination complexity

## Why We Are Doing This

Procedure and CarePlan analytics support:
- treatment pathway analysis
- care coordination analytics
- chronic care management
- operational planning
- population health
- ML feature engineering

## Final Output

`healthcare_catalog.gold.procedure_careplan_summary`

## Final Grain

One row per patient.

## Step 1 — Import PySpark Functions

### What We Are Doing
We are importing Spark SQL functions.

### Why We Are Doing This
We need aggregations, conditional logic, and feature engineering functions.

### Expected Output
Spark functions available.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Silver Procedure and CarePlan Tables

### What We Are Doing
We are reading cleaned Procedure and CarePlan tables from the Silver layer.

### Why We Are Doing This
Gold tables should be built from clean, normalized Silver tables.

### Expected Output
Procedure and CarePlan DataFrames loaded successfully.

In [0]:
procedure_df = spark.table(
    "healthcare_catalog.silver.procedure_clean"
)

careplan_df = spark.table(
    "healthcare_catalog.silver.careplan_clean"
)

print("Procedure and CarePlan Silver tables loaded successfully.")

Procedure and CarePlan Silver tables loaded successfully.


## Step 3 — Create Procedure Summary by Patient

### What We Are Doing
We are aggregating Procedure records at the patient level.

### Why We Are Doing This
Procedure counts and duration are useful indicators of treatment intensity.

### Metrics Created

- total procedures
- unique procedure types
- average procedure duration
- total procedure duration
- max procedure duration

### Expected Output
One row per patient with procedure burden metrics.

In [0]:
procedure_summary_df = procedure_df.groupBy(
    "patient_id"
).agg(
    count("*").alias("total_procedures"),

    countDistinct("procedure_description").alias("unique_procedure_count"),

    avg("procedure_duration_hours").alias("avg_procedure_duration_hours"),

    sum("procedure_duration_hours").alias("total_procedure_duration_hours"),

    max("procedure_duration_hours").alias("max_procedure_duration_hours")
)

display(procedure_summary_df)

patient_id,total_procedures,unique_procedure_count,avg_procedure_duration_hours,total_procedure_duration_hours,max_procedure_duration_hours
b0a06ead-cc42-aa48-dad6-841d4aa679fa,73,15,0.38060502283105024,27.784166666666668,0.8302777777777778
ccfc4db2-2026-7adb-3db0-33f3828140bb,44,9,0.40268308080808085,17.71805555555556,0.9591666666666666
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,65,11,0.4088632478632479,26.576111111111114,0.9661111111111111
71a8b156-760b-df6b-859e-eefc7932a526,9,2,0.24324074074074073,2.1891666666666665,0.25
76b289fd-e825-734c-8446-316f59643593,43,19,0.3610658914728681,15.52583333333333,1.6833333333333333
92fb7efc-5cfd-f8d3-927b-42f8ee099531,13,6,0.3449145299145299,4.483888888888889,0.7155555555555555
81aa7647-779f-fd6b-94cf-782e606efeb2,7,2,0.24682539682539684,1.7277777777777779,0.25
346a1435-2455-914f-c287-7b88052d05db,179,43,0.3011359404096834,53.90333333333333,0.9469444444444445
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,58,13,0.3962116858237549,22.980277777777783,0.9402777777777778
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,86,43,0.3538662790697675,30.432500000000005,1.0586111111111112


## Step 4 — Create CarePlan Summary by Patient

### What We Are Doing
We are aggregating CarePlan records at the patient level.

### Why We Are Doing This
Care plans represent longitudinal management, chronic care coordination, and treatment planning.

### Metrics Created

- total care plans
- active care plans
- completed care plans
- unique care plan categories
- average care plan duration
- total care plan duration

### Expected Output
One row per patient with care management metrics.

In [0]:
careplan_summary_df = careplan_df.groupBy(
    "patient_id"
).agg(
    count("*").alias("total_careplans"),

    sum(
        when(col("careplan_status") == "active", 1).otherwise(0)
    ).alias("active_careplans"),

    sum(
        when(col("careplan_status") == "completed", 1).otherwise(0)
    ).alias("completed_careplans"),

    countDistinct("careplan_category").alias("unique_careplan_categories"),

    avg("careplan_duration_days").alias("avg_careplan_duration_days"),

    sum("careplan_duration_days").alias("total_careplan_duration_days")
)

display(careplan_summary_df)

patient_id,total_careplans,active_careplans,completed_careplans,unique_careplan_categories,avg_careplan_duration_days,total_careplan_duration_days
b0a06ead-cc42-aa48-dad6-841d4aa679fa,7,3,4,1,253.75,1015
ccfc4db2-2026-7adb-3db0-33f3828140bb,4,2,2,1,48.0,96
31a2e8ec-69fc-8a71-3ab6-36cbdd508713,1,1,0,1,null,null
71a8b156-760b-df6b-859e-eefc7932a526,1,0,1,1,27.0,27
76b289fd-e825-734c-8446-316f59643593,7,2,5,1,15.6,78
92fb7efc-5cfd-f8d3-927b-42f8ee099531,5,1,4,1,44.25,177
81aa7647-779f-fd6b-94cf-782e606efeb2,1,0,1,1,275.0,275
346a1435-2455-914f-c287-7b88052d05db,7,3,4,1,655.0,2620
1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,2,1,1,1,12.0,12
97899f1d-9c3b-2b90-17b8-400c11ab8f0f,6,2,4,1,70.5,282


## Step 5 — Join Procedure and CarePlan Summaries

### What We Are Doing
We are joining procedure burden and care management burden into one patient-level table.

### Why We Are Doing This
Combining treatment procedures with care plans gives a better picture of patient complexity.

### Expected Output
One row per patient with treatment + care management KPIs.

In [0]:
procedure_careplan_summary_df = procedure_summary_df.join(
    careplan_summary_df,
    on="patient_id",
    how="full"
)

procedure_careplan_summary_df = procedure_careplan_summary_df.fillna(
    {
        "total_procedures": 0,
        "unique_procedure_count": 0,
        "avg_procedure_duration_hours": 0,
        "total_procedure_duration_hours": 0,
        "max_procedure_duration_hours": 0,
        "total_careplans": 0,
        "active_careplans": 0,
        "completed_careplans": 0,
        "unique_careplan_categories": 0,
        "avg_careplan_duration_days": 0,
        "total_careplan_duration_days": 0
    }
)

display(procedure_careplan_summary_df)

patient_id,total_procedures,unique_procedure_count,avg_procedure_duration_hours,total_procedure_duration_hours,max_procedure_duration_hours,total_careplans,active_careplans,completed_careplans,unique_careplan_categories,avg_careplan_duration_days,total_careplan_duration_days
965ecf4b-40d6-02e3-fe08-acd9eafc68fe,83,25,0.39478580990629175,32.767222222222216,0.9363888888888889,8,2,6,1,42.0,252
122d96b5-9b37-a7c2-7567-c3a66ae854d7,56,10,0.4190823412698413,23.468611111111112,0.8719444444444444,1,0,1,1,62.0,62
297c1550-317d-aac0-b529-042721fed414,202,49,0.2767065456545654,55.89472222222221,0.9927777777777778,5,0,5,1,443.0,2215
683fc157-3f75-3396-c034-a3fb4ec710fe,45,11,0.40828395061728395,18.372777777777777,0.9625,4,1,3,1,139.0,417
a24ffe4b-09c7-3855-e3b3-0b2983c73077,149,42,0.27418717375093216,40.85388888888889,0.8975,5,1,4,1,167.75,671
94c64aed-cca1-dc80-c180-5ba446c00df4,210,46,0.3018015873015873,63.37833333333333,0.9608333333333333,9,2,7,1,201.85714285714286,1413
767e1d64-d100-a9ec-fe05-073c6857ffa8,50,12,0.4212444444444445,21.062222222222225,0.9541666666666667,2,1,1,1,8.0,8
67c6597f-c239-a595-beac-b2941e7b54ff,52,12,0.44524572649572647,23.152777777777775,0.8680555555555556,3,1,2,1,194.5,389
2f031d4a-b070-ce15-6372-30c8fecf1164,157,42,0.3410067232837934,53.53805555555556,0.9916666666666667,8,2,6,1,1062.3333333333333,6374
29b21635-3f05-3634-1941-4122fb2471ce,60,10,0.40602314814814816,24.36138888888889,0.9686111111111111,3,1,2,1,81.0,162


## Step 6 — Create Treatment and Care Complexity Flags

### What We Are Doing
We are creating business-friendly complexity flags.

### Why We Are Doing This
Gold tables should contain dashboard-ready and ML-ready risk indicators.

### Flags Created

- high procedure burden
- active care management
- high careplan burden
- high treatment complexity

### Expected Output
New complexity flag columns.

In [0]:
procedure_careplan_summary_df = procedure_careplan_summary_df.withColumn(
    "high_procedure_burden_flag",
    when(col("total_procedures") >= 20, 1).otherwise(0)
).withColumn(
    "active_care_management_flag",
    when(col("active_careplans") > 0, 1).otherwise(0)
).withColumn(
    "high_careplan_burden_flag",
    when(col("total_careplans") >= 3, 1).otherwise(0)
).withColumn(
    "high_treatment_complexity_flag",
    when(
        (col("total_procedures") >= 20) |
        (col("total_careplans") >= 3) |
        (col("active_careplans") > 0),
        1
    ).otherwise(0)
)

display(procedure_careplan_summary_df)

patient_id,total_procedures,unique_procedure_count,avg_procedure_duration_hours,total_procedure_duration_hours,max_procedure_duration_hours,total_careplans,active_careplans,completed_careplans,unique_careplan_categories,avg_careplan_duration_days,total_careplan_duration_days,high_procedure_burden_flag,active_care_management_flag,high_careplan_burden_flag,high_treatment_complexity_flag
965ecf4b-40d6-02e3-fe08-acd9eafc68fe,83,25,0.39478580990629175,32.767222222222216,0.9363888888888889,8,2,6,1,42.0,252,1,1,1,1
122d96b5-9b37-a7c2-7567-c3a66ae854d7,56,10,0.4190823412698413,23.468611111111112,0.8719444444444444,1,0,1,1,62.0,62,1,0,0,1
297c1550-317d-aac0-b529-042721fed414,202,49,0.2767065456545654,55.89472222222221,0.9927777777777778,5,0,5,1,443.0,2215,1,0,1,1
683fc157-3f75-3396-c034-a3fb4ec710fe,45,11,0.40828395061728395,18.372777777777777,0.9625,4,1,3,1,139.0,417,1,1,1,1
a24ffe4b-09c7-3855-e3b3-0b2983c73077,149,42,0.27418717375093216,40.85388888888889,0.8975,5,1,4,1,167.75,671,1,1,1,1
94c64aed-cca1-dc80-c180-5ba446c00df4,210,46,0.3018015873015873,63.37833333333333,0.9608333333333333,9,2,7,1,201.85714285714286,1413,1,1,1,1
767e1d64-d100-a9ec-fe05-073c6857ffa8,50,12,0.4212444444444445,21.062222222222225,0.9541666666666667,2,1,1,1,8.0,8,1,1,0,1
67c6597f-c239-a595-beac-b2941e7b54ff,52,12,0.44524572649572647,23.152777777777775,0.8680555555555556,3,1,2,1,194.5,389,1,1,1,1
2f031d4a-b070-ce15-6372-30c8fecf1164,157,42,0.3410067232837934,53.53805555555556,0.9916666666666667,8,2,6,1,1062.3333333333333,6374,1,1,1,1
29b21635-3f05-3634-1941-4122fb2471ce,60,10,0.40602314814814816,24.36138888888889,0.9686111111111111,3,1,2,1,81.0,162,1,1,1,1


## Step 7 — Validate Procedure and CarePlan KPIs

### What We Are Doing
We are calculating population-level treatment and care management KPIs.

### Why We Are Doing This
This validates Gold-layer feature engineering and supports dashboard KPI development.

### Expected Output
Population-level procedure and careplan summary.

In [0]:
display(
    procedure_careplan_summary_df.select(

        avg("total_procedures").alias("avg_total_procedures"),

        avg("unique_procedure_count").alias("avg_unique_procedure_count"),

        avg("total_careplans").alias("avg_total_careplans"),

        avg("active_careplans").alias("avg_active_careplans"),

        avg("completed_careplans").alias("avg_completed_careplans"),

        sum("high_procedure_burden_flag").alias("patients_with_high_procedure_burden"),

        sum("active_care_management_flag").alias("patients_with_active_care_management"),

        sum("high_careplan_burden_flag").alias("patients_with_high_careplan_burden"),

        sum("high_treatment_complexity_flag").alias("patients_with_high_treatment_complexity")
    )
)

avg_total_procedures,avg_unique_procedure_count,avg_total_careplans,avg_active_careplans,avg_completed_careplans,patients_with_high_procedure_burden,patients_with_active_care_management,patients_with_high_careplan_burden,patients_with_high_treatment_complexity
69.41981981981982,16.46126126126126,3.299099099099099,1.38018018018018,1.9189189189189189,455,382,340,481


## Step 8 — Save Gold Procedure CarePlan Summary Table

### What We Are Doing
We are saving the patient-level procedure and careplan summary into the Gold layer.

### Why We Are Doing This
This table becomes:
- Power BI source
- care management analytics source
- treatment complexity feature store
- population health dataset

### Expected Output
A Delta table:

`healthcare_catalog.gold.procedure_careplan_summary`

In [0]:
procedure_careplan_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "healthcare_catalog.gold.procedure_careplan_summary"
    )

print("Gold procedure_careplan_summary saved successfully.")

Gold procedure_careplan_summary saved successfully.


## Step 9 — Verify Gold Tables

### What We Are Doing
We are verifying Gold tables.

### Why We Are Doing This
We want to confirm successful Gold table creation.

### Expected Output
`procedure_careplan_summary` should appear in Gold schema.

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.gold
""").show(truncate=False)

+--------+-------------------------------+-----------+
|database|tableName                      |isTemporary|
+--------+-------------------------------+-----------+
|gold    |chronic_disease_summary        |false      |
|gold    |encounter_utilization_summary  |false      |
|gold    |medication_summary             |false      |
|gold    |observation_vitals_labs_summary|false      |
|gold    |patient_summary                |false      |
|gold    |procedure_careplan_summary     |false      |
+--------+-------------------------------+-----------+

